# 02: Scope the Host

You have an agent_id from notebook 01. Now get the full host record, see the alert history from
the incident day, and find the CVEs behind what you saw in the attack chain.

This notebook uses three service classes: `Hosts` (device info), `Alerts` (same as notebook 01),
and `SpotlightVulnerabilities` (vulnerability findings from Spotlight).

In [ ]:
import os

from falconpy import Hosts, Alerts, SpotlightVulnerabilities
from rich import print as rprint
from rich.table import Table

client_id = os.environ.get("FALCON_CLIENT_ID", "")
client_secret = os.environ.get("FALCON_CLIENT_SECRET", "")

hosts = Hosts(client_id=client_id, client_secret=client_secret)
alerts = Alerts(client_id=client_id, client_secret=client_secret)
vulns = SpotlightVulnerabilities(client_id=client_id, client_secret=client_secret)


def show(title, fields):
    table = Table(title=title)
    table.add_column("field", style="cyan")
    table.add_column("value")
    for k, v in fields.items():
        table.add_row(str(k), str(v))
    rprint(table)

## The case host

Paste the agent_id from notebook 01.

In [ ]:
# TODO: paste the agent_id from notebook 01
trigger_aid = None

In [ ]:
if not trigger_aid:
    print("paste your agent_id in the cell above first")
else:
    device = hosts.get_device_details(ids=[trigger_aid])["body"]["resources"][0]

    show("Case host", {
        "hostname": device["hostname"],
        "platform": device["platform_name"],
        "os": device["os_version"],
        "local_ip": device["local_ip"],
        "last_seen": device["last_seen"],
    })

In [ ]:
if not trigger_aid:
    print("paste your agent_id in the cell above first")
else:
    trigger_resp = alerts.query_alerts_v2(
        filter=f"agent_id:'{trigger_aid}'+product:'epp'+severity_name:'Critical'",
        sort="created_timestamp.desc",
        limit=1
    )
    trigger = alerts.get_alerts_v2(composite_ids=trigger_resp["body"]["resources"])["body"]["resources"][0]

    incident_day = trigger["timestamp"][:10] + "T00:00:00Z"
    print("trigger timestamp:", trigger["timestamp"])
    print("incident day:", incident_day)

## Explore the alerts endpoint

You can filter alerts by time using `created_timestamp`. This lets you see only the alerts from
the incident day instead of old noise from earlier.

The cell below shows the difference: all alerts on this host vs just the ones from the incident
day. Notice how it combines `agent_id`, `product:'epp'`, and `created_timestamp` in one filter.

In [ ]:
if not trigger_aid:
    print("paste your agent_id first")
else:
    all_on_host = alerts.query_alerts_v2(filter=f"agent_id:'{trigger_aid}'+product:'epp'")
    print("all epp alerts on this host:", all_on_host["body"]["meta"]["pagination"]["total"])

    from_incident = alerts.query_alerts_v2(
        filter=f"agent_id:'{trigger_aid}'+product:'epp'+created_timestamp:>'{incident_day}'",
        sort="created_timestamp.asc"
    )
    print("from the incident day only:", from_incident["body"]["meta"]["pagination"]["total"])

## The attack chain

In notebook 01 you traced the full chain on this host:

1. **Entry point:** `httpd` (the Apache web server, running as `daemon`)
2. **Code execution:** a poisoned `.git/modules/.../post-checkout` hook from a cloned repository
3. **Privilege escalation:** PwnKit running `/opt/pwnkit-poc/exploit` through `pkexec`

Two packages are responsible: `git` and `policykit-1`. Now find the CVEs for those in Spotlight.

## Find the CVEs

Now switch to Spotlight (the vulnerability scanner). It stores findings for every package on
every host. This host has thousands of open findings, so you need to narrow the search.

Use `cve.is_cisa_kev:true` to show only the vulnerabilities on the CISA Known Exploited
Vulnerabilities list. Add `aid` (the agent_id) to scope it to this one host. The `facet`
parameter tells the API which extra fields to include in the response.

Look for `git` and `policykit-1` in the results. Those are the two packages from the attack
chain you just traced.

In [ ]:
# TODO: query CISA-KEV vulnerabilities on this host
# hint: vulns.query_vulnerabilities_combined(filter=f"aid:'{trigger_aid}'+...", facet=["cve", "remediation"])
vuln_resp = None  # replace this line with your query

if not vuln_resp or vuln_resp["status_code"] != 200:
    print("query failed or not filled in yet")
else:
    open_vulns = vuln_resp["body"]["resources"]

    for vuln in open_vulns:
        cve_id = vuln["cve"]["id"]
        app = vuln["apps"][0]["product_name_version"]
        actors = vuln["cve"].get("actors") or []
        print(f"{cve_id:16} {app:45} {', '.join(actors)}")

You should see `git` (CVE-2025-48384) and `policykit-1` (CVE-2021-4034) in that list, along
with their adversary attribution.

Pick the git CVE to carry forward. The cell below loops through the results and finds it for
you. Read through the code to understand what it does.

In [ ]:
# TODO: find the git CVE in the list
# loop through open_vulns, check if "git" is in the app name, and save it to picked
picked = None
for vuln in open_vulns:
    app_name = vuln["apps"][0]["product_name_version"]
    if "git" in app_name.lower():
        picked = vuln
        break

cve_id = picked["cve"]["id"]
remediation_id = picked["apps"][0]["remediation_info"]["recommended_id"]
case_actors = picked["cve"].get("actors") or []

show("Case CVE", {
    "cve_id": cve_id,
    "app": picked["apps"][0]["product_name_version"],
    "adversaries": ", ".join(case_actors),
    "remediation_id": remediation_id,
})

## Fallback

Run this only if the query above returned nothing.

In [ ]:
import json

try:
    with open("data/sample_responses/vulnerabilities_query.json") as handle:
        response = {"status_code": 200, "body": json.load(handle)}
except FileNotFoundError:
    print("sample file not found")
    response = None

if response:
    open_vulns = response["body"]["resources"]

    picked = None
    for vuln in open_vulns:
        if "git" in vuln["apps"][0]["product_name_version"].lower():
            picked = vuln
            break

    cve_id = picked["cve"]["id"]
    remediation_id = picked["apps"][0]["remediation_info"]["recommended_id"]
    case_actors = picked["cve"].get("actors") or []

    show("Fallback", {
        "cve_id": cve_id,
        "remediation_id": remediation_id,
        "adversaries": ", ".join(case_actors),
    })

## Carry to 03

Take the `cve_id` into notebook 03 to find the blast radius.

`cve_id = "<the CVE printed above>"`